# Music Annotation Project V8.2b
## Interactive Gradio Six-Band Confidence Explorer

**Purpose:** live, development-only exploration of the saved V8.2 calibrated OOF `p_print` probabilities.

This notebook does **not** retrain, refit, recalibrate, change V7.8, or create an authorised correction policy. It provides five live break-point sliders defining six confidence bands:

1. Definitely annotation
2. Probably annotation
3. Weak annotation lean
4. Weak print lean
5. Probably print
6. Definitely print

The app renders three synchronised views for the selected page:

- `classifier_opinion_on_page.png` — annotation-facing pixels become white, print-facing pixels become black, and the two central bands become light/dark grey.
- `extreme_state_review_preview.png` — definite annotation becomes white, definite print becomes black, and the four middle bands remain coloured.
- `six_state_confidence_overlay.png` — the full six-colour overlay on the annotated page.

Use **Previous / Next** or the page selector to move through the available annotated development pages. Shared zoom and pan controls crop and enlarge the same part of all three views. **Refresh** rebuilds the current view. **Save snapshot** writes the three full-resolution images and a JSON record of the breakpoints to Google Drive.

**Important:** every threshold in this UI is diagnostic. Moving a slider does not alter the saved V8.2 model, calibrator, OOF probabilities, or registered operational rule.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q 'gradio>=5.0,<6' opencv-python-headless pandas numpy pyarrow pillow


## 1. Configuration and source V8.2 run

If required, set `SOURCE_V82_RUN_OVERRIDE` to one completed V8.2 run. Otherwise the notebook selects the most recently modified complete run.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib
import cv2
import numpy as np
import pandas as pd
import gradio as gr

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT = ROOT / 'observed_print_classifier_v8_2'
EXPORT_ROOT = ROOT / 'observed_print_classifier_v8_2b_gradio_explorer'
SOURCE_V82_RUN_OVERRIDE = None

REQUIRED = [
    'experiment_identity.json',
    'tables/all_eligible_oof_predictions.parquet',
    'whole_pages',
]

def complete_runs(root):
    if not root.exists():
        return []
    return sorted(
        [p for p in root.iterdir() if p.is_dir() and all((p / r).exists() for r in REQUIRED)],
        key=lambda p: p.stat().st_mtime,
    )

if SOURCE_V82_RUN_OVERRIDE is not None:
    SOURCE = Path(SOURCE_V82_RUN_OVERRIDE)
    missing = [r for r in REQUIRED if not (SOURCE / r).exists()]
    if missing:
        raise RuntimeError(f'Incomplete source override: {missing}')
else:
    runs = complete_runs(V82_ROOT)
    if not runs:
        raise RuntimeError(f'No complete V8.2 run found below {V82_ROOT}')
    SOURCE = runs[-1]

pred_path = SOURCE / 'tables' / 'all_eligible_oof_predictions.parquet'
full = pd.read_parquet(pred_path)
required_columns = {'page','y','x','p_print'}
missing = required_columns - set(full.columns)
if missing:
    raise RuntimeError(f'Prediction table missing columns: {sorted(missing)}')

for c in ['y','x']:
    full[c] = pd.to_numeric(full[c], errors='raise').astype(int)
full['p_print'] = pd.to_numeric(full['p_print'], errors='raise').astype(np.float64)
if not np.isfinite(full.p_print.to_numpy()).all():
    raise RuntimeError('Non-finite p_print values found.')
if full.duplicated(['page','y','x']).any():
    raise RuntimeError('Duplicate page/y/x rows found.')

# Keep only pages with a readable annotated source image.
pages = []
for page in sorted(full.page.astype(str).unique()):
    p = SOURCE / 'whole_pages' / page / 'observed.png'
    if p.exists() and cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) is not None:
        pages.append(page)
if not pages:
    raise RuntimeError('No readable observed.png pages found in the selected V8.2 run.')

page_rows = {page: full[full.page.astype(str) == page].copy() for page in pages}
page_to_index = {p:i for i,p in enumerate(pages)}

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE)
print('Pages available:', len(pages))
print('Eligible rows:', len(full))

## 2. Rendering helpers

The five breakpoints are automatically sorted before rendering, so the six bands always remain ordered even while sliders are being moved quickly.

In [ ]:
# RGB colours for Gradio/Pillow output.
COLOURS = np.array([
    [200,   0,   0],   # definite annotation — dark red
    [255, 125,   0],   # probable annotation — orange
    [ 25, 100, 230],   # weak annotation lean — blue
    [  0, 205, 210],   # weak print lean — cyan
    [120, 255,   0],   # probable print — lime
    [  0, 145,   0],   # definite print — dark green
], dtype=np.uint8)

STATE_NAMES = [
    'Definitely annotation',
    'Probably annotation',
    'Weak annotation lean',
    'Weak print lean',
    'Probably print',
    'Definitely print',
]

def normalise_breaks(a, b, c, d, e):
    raw = np.array([a,b,c,d,e], dtype=float) / 100.0
    ordered = np.sort(np.clip(raw, 0.0, 1.0))
    # Avoid zero-width bands caused by equal sliders.
    eps = 1e-6
    for i in range(1, len(ordered)):
        if ordered[i] <= ordered[i-1]:
            ordered[i] = min(1.0, ordered[i-1] + eps)
    return ordered

def classify_probabilities(prob, breaks):
    # Returns 0..5.
    return np.digitize(prob, breaks, right=False).astype(np.uint8)

def dilated_state_masks(shape, ys, xs, states, radius):
    masks = []
    k = None if radius <= 0 else np.ones((2*radius+1, 2*radius+1), np.uint8)
    for state_id in range(6):
        m = np.zeros(shape, np.uint8)
        hit = states == state_id
        m[ys[hit], xs[hit]] = 1
        if k is not None:
            m = cv2.dilate(m, k)
        masks.append(m > 0)
    return masks

def add_legend_rgb(image, title, breaks):
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    h, w = image.shape[:2]
    banner = max(110, int(h * 0.065))
    canvas = np.full((h + banner, w, 3), 255, np.uint8)
    canvas[banner:] = image
    scale = max(0.42, min(0.78, w / 1900.0))
    cv2.putText(canvas, title, (12, int(banner*0.28)), cv2.FONT_HERSHEY_SIMPLEX,
                scale, (25,25,25), 1, cv2.LINE_AA)
    lows = [0.0, *breaks.tolist()]
    highs = [*breaks.tolist(), 1.0]
    labels = [f'{name} [{lo:.3f}-{hi:.3f}{"]" if i==5 else ")"}'
              for i,(name,lo,hi) in enumerate(zip(STATE_NAMES,lows,highs))]
    # Two rows of three items to remain readable on narrow pages.
    positions = [(0,0),(1,0),(2,0),(0,1),(1,1),(2,1)]
    col_w = max(240, w // 3)
    for i, (col,row) in enumerate(positions):
        x = 12 + col * col_w
        y = int(banner * (0.55 + 0.32*row))
        colour = tuple(int(v) for v in COLOURS[i])
        cv2.rectangle(canvas, (x,y-15), (x+18,y+3), colour, -1)
        cv2.putText(canvas, labels[i], (x+25,y+2), cv2.FONT_HERSHEY_SIMPLEX,
                    scale*0.62, (25,25,25), 1, cv2.LINE_AA)
    return canvas

def zoom_pan(image, zoom, pan_x, pan_y):
    if image is None or zoom <= 1.001:
        return image
    h, w = image.shape[:2]
    crop_w = max(32, int(round(w / zoom)))
    crop_h = max(32, int(round(h / zoom)))
    max_x = max(0, w - crop_w)
    max_y = max(0, h - crop_h)
    x0 = int(round(max_x * (pan_x / 100.0)))
    y0 = int(round(max_y * (pan_y / 100.0)))
    crop = image[y0:y0+crop_h, x0:x0+crop_w]
    return cv2.resize(crop, (w, h), interpolation=cv2.INTER_NEAREST)

def render_full(page, a, b, c, d, e, alpha, radius):
    if page not in page_rows:
        raise ValueError(f'Unknown page: {page}')
    observed = cv2.imread(str(SOURCE/'whole_pages'/page/'observed.png'), cv2.IMREAD_GRAYSCALE)
    if observed is None:
        raise RuntimeError(f'Could not read observed page for {page}')
    g = page_rows[page]
    ys = g.y.to_numpy(int)
    xs = g.x.to_numpy(int)
    prob = g.p_print.to_numpy(float)
    if (ys < 0).any() or (xs < 0).any() or (ys >= observed.shape[0]).any() or (xs >= observed.shape[1]).any():
        raise RuntimeError(f'{page}: prediction coordinate outside image')

    breaks = normalise_breaks(a,b,c,d,e)
    states = classify_probabilities(prob, breaks)
    masks = dilated_state_masks(observed.shape, ys, xs, states, int(radius))
    base_rgb = cv2.cvtColor(observed, cv2.COLOR_GRAY2RGB)

    # 1. Six-state colour overlay.
    colour = base_rgb.copy()
    active = np.zeros(observed.shape, bool)
    for state_id, mask in enumerate(masks):
        colour[mask] = COLOURS[state_id]
        active |= mask
    overlay = base_rgb.copy()
    blended = cv2.addWeighted(base_rgb, 1.0-float(alpha), colour, float(alpha), 0)
    overlay[active] = blended[active]
    six_state = add_legend_rgb(overlay, f'{page} — live six-state annotation-to-print confidence', breaks)

    # 2. Opinion on page: 0/1 white, 2 light grey, 3 dark grey, 4/5 black.
    opinion = observed.copy()
    opinion_values = [255, 255, 190, 95, 0, 0]
    for state_id, mask in enumerate(masks):
        opinion[mask] = opinion_values[state_id]
    opinion_rgb = cv2.cvtColor(opinion, cv2.COLOR_GRAY2RGB)
    opinion_view = add_legend_rgb(
        opinion_rgb,
        f'{page} — classifier opinion on page (white / light grey / dark grey / black)',
        breaks,
    )

    # 3. Extreme review: state 0 white, state 5 black; inner four remain coloured.
    extreme = base_rgb.copy()
    extreme[masks[0]] = (255,255,255)
    for state_id in [1,2,3,4]:
        extreme[masks[state_id]] = COLOURS[state_id]
    extreme[masks[5]] = (0,0,0)
    extreme_view = add_legend_rgb(
        extreme,
        f'{page} — extreme review preview (outer states resolved; four middle bands retained)',
        breaks,
    )

    counts = np.bincount(states, minlength=6)
    ranges = [0.0, *breaks.tolist(), 1.0]
    status_lines = [
        f'**Page:** {page} ({page_to_index[page]+1} of {len(pages)})',
        f'**Eligible pixels:** {len(g):,}',
        f'**Display radius:** {int(radius)} px · **Overlay alpha:** {float(alpha):.2f}',
        '',
        '**Live band counts:**',
    ]
    for i in range(6):
        close = ']' if i == 5 else ')'
        status_lines.append(
            f'- {STATE_NAMES[i]} [{ranges[i]:.3f}–{ranges[i+1]:.3f}{close}: **{int(counts[i]):,}**'
        )
    status = '\n'.join(status_lines)
    metadata = {
        'page': page,
        'breakpoints_probability': breaks.tolist(),
        'breakpoints_percent': (breaks*100).tolist(),
        'state_names': STATE_NAMES,
        'state_counts': {STATE_NAMES[i]: int(counts[i]) for i in range(6)},
        'overlay_alpha': float(alpha),
        'display_radius': int(radius),
        'source_v82_run': str(SOURCE),
        'diagnostic_only': True,
    }
    return opinion_view, extreme_view, six_state, status, metadata

def render_live(page, a, b, c, d, e, alpha, radius, zoom, pan_x, pan_y):
    opinion, extreme, six_state, status, metadata = render_full(page,a,b,c,d,e,alpha,radius)
    return (
        zoom_pan(opinion, zoom, pan_x, pan_y),
        zoom_pan(extreme, zoom, pan_x, pan_y),
        zoom_pan(six_state, zoom, pan_x, pan_y),
        status,
        metadata,
    )

def step_page(page, delta):
    idx = page_to_index.get(page, 0)
    return pages[(idx + int(delta)) % len(pages)]

def save_snapshot(page, a, b, c, d, e, alpha, radius):
    opinion, extreme, six_state, status, metadata = render_full(page,a,b,c,d,e,alpha,radius)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    folder = EXPORT_ROOT / f'{stamp}_{page}_bands'
    folder.mkdir(parents=True, exist_ok=False)
    # Convert RGB to BGR for OpenCV writing.
    cv2.imwrite(str(folder/'classifier_opinion_on_page.png'), cv2.cvtColor(opinion, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(folder/'extreme_state_review_preview.png'), cv2.cvtColor(extreme, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(folder/'six_state_confidence_overlay.png'), cv2.cvtColor(six_state, cv2.COLOR_RGB2BGR))
    metadata['saved_utc'] = datetime.now(timezone.utc).isoformat()
    (folder/'band_experiment.json').write_text(json.dumps(metadata, indent=2))
    return f'Saved snapshot to `{folder}`'

## 3. Launch the live Gradio explorer

Move any breakpoint slider to rebuild all three images. The notebook sorts breakpoints before rendering, so no invalid order can crash the UI. Zoom/pan is implemented as a shared high-resolution crop and enlargement, giving the same viewport across all three outputs.

In [ ]:
DEFAULT_PAGE = pages[0]

with gr.Blocks(title='V8.2b Six-Band Confidence Explorer') as demo:
    gr.Markdown(
        '# V8.2b Interactive Six-Band Confidence Explorer\n'
        'Development-only visualisation of saved V8.2 OOF probabilities. '
        '**No model, calibration, registered decision, or corrected page is changed.**'
    )

    with gr.Row():
        previous_btn = gr.Button('◀ Previous', variant='secondary')
        page_dd = gr.Dropdown(choices=pages, value=DEFAULT_PAGE, label='Annotated page', filterable=True)
        next_btn = gr.Button('Next ▶', variant='secondary')
        refresh_btn = gr.Button('↻ Refresh', variant='primary')

    gr.Markdown('### Six confidence bands — five live breakpoints (%)')
    with gr.Row():
        t1 = gr.Slider(0, 100, value=10, step=1, label='1. Definite annotation / probable annotation')
        t2 = gr.Slider(0, 100, value=35, step=1, label='2. Probable annotation / weak annotation lean')
        t3 = gr.Slider(0, 100, value=50, step=1, label='3. Weak annotation lean / weak print lean')
    with gr.Row():
        t4 = gr.Slider(0, 100, value=65, step=1, label='4. Weak print lean / probable print')
        t5 = gr.Slider(0, 100, value=90, step=1, label='5. Probable print / definite print')

    with gr.Row():
        alpha = gr.Slider(0.10, 1.00, value=0.62, step=0.02, label='Colour overlay alpha')
        radius = gr.Slider(0, 3, value=1, step=1, label='Visibility radius (pixels)')

    gr.Markdown('### Shared zoom and pan')
    with gr.Row():
        zoom = gr.Slider(1.0, 8.0, value=1.0, step=0.25, label='Zoom')
        pan_x = gr.Slider(0, 100, value=50, step=1, label='Pan left ↔ right (%)')
        pan_y = gr.Slider(0, 100, value=50, step=1, label='Pan top ↔ bottom (%)')

    band_status = gr.Markdown()
    live_metadata = gr.JSON(label='Current diagnostic state', open=False)

    with gr.Tabs():
        with gr.Tab('Classifier opinion on page'):
            opinion_output = gr.Image(type='numpy', label='classifier_opinion_on_page.png', interactive=False)
        with gr.Tab('Extreme review preview'):
            extreme_output = gr.Image(type='numpy', label='extreme_state_review_preview.png', interactive=False)
        with gr.Tab('Six-state confidence overlay'):
            six_state_output = gr.Image(type='numpy', label='six_state_confidence_overlay.png', interactive=False)

    with gr.Row():
        save_btn = gr.Button('💾 Save current full-resolution snapshot')
        save_status = gr.Markdown()

    render_inputs = [page_dd, t1, t2, t3, t4, t5, alpha, radius, zoom, pan_x, pan_y]
    render_outputs = [opinion_output, extreme_output, six_state_output, band_status, live_metadata]

    def go_previous(page):
        return step_page(page, -1)

    def go_next(page):
        return step_page(page, 1)

    previous_btn.click(go_previous, inputs=page_dd, outputs=page_dd)
    next_btn.click(go_next, inputs=page_dd, outputs=page_dd)

    # Any page, threshold, zoom or pan change redraws the three synchronised views.
    for control in render_inputs:
        control.change(render_live, inputs=render_inputs, outputs=render_outputs, show_progress='hidden')

    refresh_btn.click(render_live, inputs=render_inputs, outputs=render_outputs, show_progress='minimal')
    save_btn.click(
        save_snapshot,
        inputs=[page_dd, t1, t2, t3, t4, t5, alpha, radius],
        outputs=save_status,
        show_progress='minimal',
    )

    demo.load(render_live, inputs=render_inputs, outputs=render_outputs, show_progress='minimal')

# In Colab, share=True supplies the temporary Gradio link.
demo.queue(default_concurrency_limit=2).launch(share=True, debug=True)

## Interpretation guardrail

This explorer deliberately separates **diagnostic confidence bands** from the registered V8 decision rule. Slider positions are not validated operating points. Use the UI to inspect spatial behaviour and formulate a subsequent, separately declared experiment; do not treat a visually pleasing threshold arrangement as promotion evidence.